# FightWatch — đối chứng quan hệ người + chuyển động
Chỉ dùng cache public AIRTLab/RLVS đã có. Giữ nguyên split nhóm biểu diễn, đóng băng ngưỡng trước diagnostic. Hai video iPhone giữ tại máy, không upload/train. Giới hạn **toàn phiên GPU 25 phút**, tính cả cài đặt/staging; watchdog tự giải phóng phiên sớm hơn. Không tự thay model local và không cam kết mọi tình huống.

In [ ]:
import time, threading, json, os, sys, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import runtime, drive
if 'DEADLINE' not in globals():
    STARTED = time.monotonic()
    DEADLINE = STARTED + 23 * 60
    def release_budget():
        print("GPU wall-clock budget reached; releasing runtime.", flush=True)
        runtime.unassign()
    WATCHDOG = threading.Timer(23 * 60, release_budget)
    WATCHDOG.daemon = True
    WATCHDOG.start()
print("GPU watchdog armed; rerunning does not reset the allocation budget.", flush=True)
try:
    drive.mount('/content/drive', timeout_ms=180000)
except BaseException:
    runtime.unassign()
    raise
ROOT = Path('/content/drive/MyDrive/SENTINEL')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-pair-motion-code')
SOURCE = ROOT / 'artifacts/fight_fusion_v1'
OUTPUT = ROOT / 'artifacts/fight_pair_motion_v1'
CACHE = Path('/content/fightwatch-pair-motion-cache')
MOTION = Path('/content/fightwatch-pair-motion-flow')
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Ready:", OUTPUT)


## Chạy đối chứng và tự ngắt GPU
Chỉ bỏ autosave notebook tracked trong repo Drive, không bỏ sửa code khác; pull fast-forward main. RGB frozen control và pair+motion dùng cùng split/seed/sampling. Dừng sớm khi development không tiến triển; test là diagnostic lặp lại. Checkpoint/log/report lưu trực tiếp trên Drive trước khi ngắt. Nếu lỗi không tự chạy lại.

In [ ]:
try:
    assert not (OUTPUT / 'run.json').exists(), "Experiment already started; inspect saved report, no automatic retry."
    print('SYNC: restoring notebook autosaves only', flush=True)
    subprocess.run(['git','-C',str(REPO),'restore','--','colab'], check=True, timeout=60)
    dirty = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'], text=True, timeout=60)
    assert not dirty.strip(), dirty
    print('SYNC: pulling main', flush=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True, timeout=90)
    REVISION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
    print('SYNC: staging code on local Colab disk', flush=True)
    shutil.copytree(REPO, LOCAL, dirs_exist_ok=True, ignore=shutil.ignore_patterns('.git','.venv','logs','uploads','models','__pycache__','.pytest_cache'))
    os.chdir(LOCAL)
    from scripts.colab_process import run_logged
    def remaining(reserve=0):
        value = DEADLINE - time.monotonic() - reserve
        assert value > 0, "GPU wall-clock budget exhausted"
        return value
    (OUTPUT / 'provenance.json').write_text(json.dumps({'revision':REVISION,'protocol':'public frozen-feature ablation; no phone upload','wall_clock_cap_seconds':1380}, indent=2))
    print('SETUP: installing runtime dependencies', flush=True)
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'], OUTPUT/'install.log', min(240, remaining(240)), revision=REVISION)
    import torch
    assert torch.cuda.is_available(), "GPU required; do not silently run long on CPU"
    print("GPU:", torch.cuda.get_device_name(0), "remaining seconds:", round(remaining()), flush=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    archive_path = Path('/content/fightwatch-window-cache.zip')
    print('CACHE: copying and extracting existing public windows', flush=True)
    shutil.copyfile(SOURCE/'window_cache.zip', archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        assert all('/' not in member.filename and '\\' not in member.filename and member.filename.endswith('.npz') for member in archive.infolist()), "Expected flat public NPZ cache"
        assert sum(member.file_size for member in archive.infolist()) < 12 * 1024**3, "Oversized source cache"
        archive.extractall(CACHE)
    shutil.copyfile(SOURCE/'source_manifest.json', OUTPUT/'source_manifest.json')
    baseline = Path('/content/pair-motion-baseline.pt')
    shutil.copyfile(ROOT/'artifacts/fight_large_v1/best.pt', baseline)
    pose = Path('/content/yolov8n-pose.pt')
    with zipfile.ZipFile(SOURCE/'rgb_aux/sentinel_fight_only_runtime.zip') as archive:
        pose.write_bytes(archive.read('models/yolov8n-pose.pt'))
    budget = min(18, remaining(90)/60)
    assert budget >= 6, "Insufficient remaining allocation for a fair two-variant pilot"
    command = [sys.executable,'-m','scripts.fight_pair_motion_train','--manifest',str(OUTPUT/'source_manifest.json'),'--cache',str(CACHE),'--motion-cache',str(MOTION),'--baseline',str(baseline),'--pose-weights',str(pose),'--output',str(OUTPUT),'--budget-minutes',str(budget)]
    run_logged(command, OUTPUT/'training.log', remaining(60), revision=REVISION)
    report = json.loads((OUTPUT/'run.json').read_text())
    print("FINAL:", report['status'], "selected:", report.get('recommended_by_development_only'), "pair progress:", report.get('pair_progress_over_control'), flush=True)
    for name, variant in report['variants'].items():
        print(name, "epochs:",variant['epochs'], "development:",variant['development'], "public diagnostics:",variant.get('test_by_dataset'), flush=True)
    if remaining() > 100:
        partial = OUTPUT / 'motion_cache.partial.zip'
        with zipfile.ZipFile(partial, 'w', zipfile.ZIP_STORED) as archive:
            for cached in sorted(MOTION.glob('*.npz')):
                if remaining() < 75:
                    break
                archive.write(cached, cached.name)
        partial.replace(OUTPUT/'motion_cache.zip')
    print("Saved:", OUTPUT, "wall-clock seconds:",round(time.monotonic()-STARTED), flush=True)
except BaseException as error:
    (OUTPUT/'notebook_failure.json').write_text(json.dumps({'error':str(error),'elapsed_seconds':time.monotonic()-STARTED},indent=2))
    raise
finally:
    print("Releasing GPU now; Drive artifacts retained. Local CPU/private diagnostics are separate.",flush=True)
    runtime.unassign()
